# 04 Multi-Head Attention & HuggingFace BERT Extractions

Loading a pretrained HuggingFace `bert-base-uncased` model, tokenizing a real text prompt, extracting live attention weights, and plotting individual head specialization patterns.

## Part 1: Loading Pretrained BERT Model & Tokenizing Text
Extracting attention weights from Layer 11.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import matplotlib.pyplot as plt

plt.style.use('dark_background')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model.eval()

text = 'The animal didn\'t cross the street because it was too tired'
inputs = tokenizer(text, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

with torch.no_grad():
    outputs = model(**inputs)

attn_layer_11 = outputs.attentions[-1][0].numpy() # (12, N, N)
print(f'Extracted {attn_layer_11.shape[0]} attention heads across {len(tokens)} tokens.')

## Part 2: Plotting 4 Distinct Attention Head Patterns
Visualizing Head 0 (Positional), Head 3 (Coreference resolution: 'it' -> 'animal'), Head 6 (Verb-Object), and Head 9 (Global Context).

In [ ]:
N = len(tokens)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
heads = [
    (0, 'BERT Layer 11 - Head 0 (Positional / Delimiters)', 'cool'),
    (3, 'BERT Layer 11 - Head 3 (Coreference: it -> animal)', 'spring'),
    (6, 'BERT Layer 11 - Head 6 (Syntactic Verb-Object)', 'plasma'),
    (9, 'BERT Layer 11 - Head 9 (Global Sentence Context)', 'Blues')
]
for idx, (h_idx, title, cmap) in enumerate(heads):
    ax = axes[idx // 2, idx % 2]
    im = ax.imshow(attn_layer_11[h_idx], cmap=cmap)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.75)
plt.tight_layout()
plt.show()